# RESA raw GCS corpus statistics

This Colab notebook produces paper-ready aggregate statistics for **every unique session stored below** `gs://miamioh-resa-data/CapstoneData/raw/sessions/`. It inventories the live object tree, downloads and validates every `meta_data.json`, and writes the exact per-session and aggregate evidence tables to GCS.

Definitions used here:

- **Session**: one raw session root containing a `meta_data.json`/`metadata.json`. The metadata `session` value must be unique; collisions fail rather than silently double-counting.
- **Raw radar frame**: one captured ADC frame. `radar.total_frames` is the recorder's count and is independently checked against the chunked DCA container.
- **Radar point/detection**: one post-CFAR/AoA Cartesian return. It is *not* an ADC sample. The optional point-count pass runs the pinned DCA DSP pipeline across every session and only reports a corpus total when all eligible sessions succeed.

The generated `aggregate_statistics.json`, `raw_session_statistics.csv`, `coverage_report.json`, and point-count evidence are the figures to cite; do not copy partially completed notebook output into a paper.

In [ ]:
# Colab authentication and common imports.  This notebook is read-only except for its
# explicitly versioned evidence output prefix.
from __future__ import annotations

import csv
import hashlib
import json
import os
import re
import shutil
import struct
import subprocess
import sys
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import auth
    auth.authenticate_user()

def run(cmd: list[str], *, capture: bool = False, check: bool = True) -> str:
    print('$', ' '.join(map(str, cmd)))
    completed = subprocess.run(
        cmd, check=check, text=True,
        stdout=subprocess.PIPE if capture else None,
        stderr=subprocess.STDOUT if capture else None,
    )
    return completed.stdout or '' if capture else ''

def gcloud_cp(source: str | Path, destination: str | Path) -> None:
    run(['gcloud', 'storage', 'cp', str(source), str(destination)])

def gcloud_ls_recursive(uri: str) -> list[str]:
    output = run(['gcloud', 'storage', 'ls', '--recursive', uri], capture=True)
    return [line.strip() for line in output.splitlines() if line.strip().startswith('gs://')]


In [ ]:
# Run configuration.  Keep RUN_ID unchanged when resuming a failed run so the evidence
# directory remains one auditable record; use a fresh ID for a new paper snapshot.
PROJECT_ID = 'fluent-webbing-496616-u8'
BUCKET = 'miamioh-resa-data'
RAW_ROOT = f'gs://{BUCKET}/CapstoneData/raw/sessions'
RUN_ID = 'raw_corpus_statistics_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUTPUT_ROOT = f'gs://{BUCKET}/CapstoneData/colab_outputs/raw_corpus_statistics/{RUN_ID}'
WORK_ROOT = Path('/content/raw_corpus_statistics')
METADATA_DIR = WORK_ROOT / 'metadata'
STAGING_DIR = WORK_ROOT / 'staging'

# A full raw-frame verification downloads one .bin at a time, so it is exact while keeping
# Colab disk use bounded.  It is required before publishing a frame total.
VERIFY_ALL_RAW_FRAME_COUNTS = True
# A point count is expensive because it runs DCA decode -> RD -> CFAR -> AoA for every frame.
# Leave true for a final paper run.  Set false only for an inventory/frame-count dry run.
RUN_DETECTION_POINT_COUNT = True
# This is the current production code tree: a full push of the RESA_mmWave repo.  The old
# 'CapstoneData/code/canon' tree was archived to 'CapstoneData/archive/code_legacy_2026-08-29/'
# on 2026-08-29 and no longer exists -- do not revert to it.  Pin this to an immutable
# revision/prefix before the final paper run if the canonical tree is expected to change.
DSP_CODE_ROOT = f'gs://{BUCKET}/CapstoneData/code_v2/RESA_mmWave'
DSP_CONFIG_URI = f'{DSP_CODE_ROOT}/config/profile_objdet.cfg'
MAX_METADATA_WORKERS = 16

run(['gcloud', 'config', 'set', 'project', PROJECT_ID])
run(['gcloud', 'config', 'set', 'storage/process_count', '4'])
run(['gcloud', 'config', 'set', 'storage/thread_count', '16'])
for path in (WORK_ROOT, METADATA_DIR, STAGING_DIR):
    path.mkdir(parents=True, exist_ok=True)
print(f'Raw scope: {RAW_ROOT}/')
print(f'Evidence output: {OUTPUT_ROOT}/')

## 1. Live GCS inventory and session topology

Discovery is based on the live object listing, not the local manifest: the requested scope is exactly the objects currently under `raw/sessions`. A session root is located by the first path component matching `session_*`; anything outside that layout is retained in the coverage report as an anomaly.

In [ ]:
SESSION_RE = re.compile(r'^session_.+')

def relative_parts(uri: str) -> list[str]:
    prefix = RAW_ROOT.rstrip('/') + '/'
    if not uri.startswith(prefix):
        raise ValueError(f'Object outside requested scope: {uri}')
    return [p for p in uri[len(prefix):].split('/') if p]

def session_root_from_object(uri: str) -> tuple[str, str, str] | None:
    parts = relative_parts(uri)
    indices = [i for i, part in enumerate(parts) if SESSION_RE.fullmatch(part)]
    if len(indices) != 1:
        return None
    index = indices[0]
    if index < 1:
        return None
    session_id = parts[index]
    dataset_id = '/'.join(parts[:index])
    root = RAW_ROOT.rstrip('/') + '/' + '/'.join(parts[:index + 1])
    return root, dataset_id, session_id

objects = gcloud_ls_recursive(RAW_ROOT)
if not objects:
    raise RuntimeError(f'No objects found below {RAW_ROOT}; check authentication and scope.')

(WORK_ROOT / 'raw_object_inventory.txt').write_text('\n'.join(objects) + '\n')
sessions: dict[str, dict[str, Any]] = {}
unassigned_objects: list[str] = []
for uri in objects:
    located = session_root_from_object(uri)
    if located is None:
        unassigned_objects.append(uri)
        continue
    root, dataset_id, session_id = located
    record = sessions.setdefault(root, {
        'root_uri': root, 'dataset_id': dataset_id, 'folder_session_id': session_id, 'objects': [],
    })
    record['objects'].append(uri)

for record in sessions.values():
    names = {Path(uri).name.lower(): uri for uri in record['objects']}
    record['metadata_uris'] = [uri for uri in record['objects'] if Path(uri).name.lower() in {'meta_data.json', 'metadata.json'}]
    record['bin_uris'] = [uri for uri in record['objects'] if uri.lower().endswith('.bin')]
    expected_name = record['folder_session_id'] + '.bin'
    exact = [uri for uri in record['bin_uris'] if Path(uri).name == expected_name]
    record['selected_bin_uri'] = exact[0] if len(exact) == 1 else (record['bin_uris'][0] if len(record['bin_uris']) == 1 else None)
    record['color_timestamp_uri'] = names.get(record['folder_session_id'].lower() + '_color_timestamps.csv')
    record['radar_timestamp_uri'] = names.get(record['folder_session_id'].lower() + '_radar_timestamps.csv')
    record['depth_object_count'] = sum('/depth/' in uri.lower() for uri in record['objects'])

print(f'GCS objects: {len(objects):,}')
print(f'Session roots: {len(sessions):,}')
dataset_prefixes = {r['dataset_id'] for r in sessions.values()}
print(f'Dataset prefixes: {len(dataset_prefixes):,}')
print(f'Objects outside session layout: {len(unassigned_objects):,}')


In [ ]:
# Download every metadata object concurrently.  Metadata files are small; bins are deliberately
# not bulk-downloaded because all later raw analysis stages are rolling and disk-bounded.
metadata_jobs = []
for ordinal, record in enumerate(sessions.values()):
    if len(record['metadata_uris']) == 1:
        folder_id = record['folder_session_id']
        local = METADATA_DIR / f'{ordinal:06d}_{folder_id}.json'
        metadata_jobs.append((record, record['metadata_uris'][0], local))

def download_metadata(job: tuple[dict[str, Any], str, Path]) -> tuple[dict[str, Any], Path, str | None]:
    record, uri, local = job
    try:
        gcloud_cp(uri, local)
        return record, local, None
    except Exception as exc:
        return record, local, repr(exc)

metadata_download_errors = []
with ThreadPoolExecutor(max_workers=MAX_METADATA_WORKERS) as pool:
    futures = [pool.submit(download_metadata, job) for job in metadata_jobs]
    for future in as_completed(futures):
        record, local, error = future.result()
        if error:
            metadata_download_errors.append({'root_uri': record['root_uri'], 'error': error})
            continue
        try:
            record['metadata'] = json.loads(local.read_text(encoding='utf-8'))
            record['metadata_sha256'] = hashlib.sha256(local.read_bytes()).hexdigest()
        except Exception as exc:
            metadata_download_errors.append({'root_uri': record['root_uri'], 'error': f'invalid JSON: {exc!r}'})

def nested_value(data: dict[str, Any], *path: str) -> Any:
    current: Any = data
    for key in path:
        if not isinstance(current, dict) or key not in current:
            return None
        current = current[key]
    return current

def nonnegative_int(value: Any) -> int | None:
    try:
        number = int(value)
        return number if number >= 0 else None
    except (TypeError, ValueError):
        return None

for record in sessions.values():
    meta = record.get('metadata')
    if not isinstance(meta, dict):
        continue
    record['metadata_session_id'] = str(meta.get('session') or '') or None
    record['recorder'] = meta.get('recorder')
    record['bin_format_version'] = nonnegative_int(meta.get('bin_format_version'))
    record['radar_frames_metadata'] = nonnegative_int(nested_value(meta, 'radar', 'total_frames'))
    record['radar_bytes_metadata'] = nonnegative_int(nested_value(meta, 'radar', 'total_bytes'))
    record['duration_s'] = nested_value(meta, 'actual_session_duration_s')
    record['depth_frames_metadata'] = nonnegative_int(nested_value(meta, 'depth', 'num_frames'))
    record['radar_frame_period_ms'] = nested_value(meta, 'radar_frame_period_ms')

by_metadata_id: dict[str, list[dict[str, Any]]] = defaultdict(list)
for record in sessions.values():
    if record.get('metadata_session_id'):
        by_metadata_id[record['metadata_session_id']].append(record)
duplicate_metadata_ids = {key: value for key, value in by_metadata_id.items() if len(value) > 1}

metadata_parsed_count = sum('metadata' in r for r in sessions.values())
print(f'Metadata downloaded and parsed: {metadata_parsed_count:,}/{len(sessions):,}')
print(f'Metadata download/parse failures: {len(metadata_download_errors):,}')
print(f'Duplicate metadata session IDs: {len(duplicate_metadata_ids):,}')


## 2. Independent raw-frame validation

Version-3 recorder files are chunked DCA containers with a `<QII` (timestamp, payload length, auxiliary) header per captured frame. This pass streams each `.bin` to temporary local storage, validates the container without decoding ADC samples, compares its chunk count with `radar.total_frames`, and removes the local file before moving on. A mismatch is a hard coverage failure.

In [ ]:
CHUNK_HEADER = struct.Struct('<QII')

def count_chunked_dca_frames(bin_path: Path) -> tuple[int, int]:
    # Returns (frames, ADC payload bytes).  One chunk is one recorder frame for the supported
    # v3 capture format.  Do not use this for flat legacy files.
    frame_count = 0
    payload_bytes = 0
    file_size = bin_path.stat().st_size
    with bin_path.open('rb') as handle:
        while True:
            offset = handle.tell()
            header = handle.read(CHUNK_HEADER.size)
            if not header:
                break
            if len(header) != CHUNK_HEADER.size:
                raise ValueError(f'truncated header at byte {offset}')
            _, payload_size, _ = CHUNK_HEADER.unpack(header)
            if payload_size <= 0 or payload_size > 16 * 1024 * 1024:
                raise ValueError(f'invalid payload size {payload_size} at byte {offset}')
            next_offset = handle.tell() + payload_size
            if next_offset > file_size:
                raise ValueError(f'truncated payload at byte {offset}')
            handle.seek(payload_size, os.SEEK_CUR)
            frame_count += 1
            payload_bytes += payload_size
    return frame_count, payload_bytes

# Use the project's DCA parser for frame enumeration.  Unlike a simple header count, this
# correctly handles both chunked recordings and contiguous legacy captures.
dsp_dir = WORK_ROOT / 'dsp'
dsp_dir.mkdir(parents=True, exist_ok=True)
dsp_script = dsp_dir / 'adc_to_pointcloud_v6.py'
dsp_config = dsp_dir / 'profile_objdet.cfg'
gcloud_cp(f'{DSP_CODE_ROOT}/branch1/processing/adc_to_pointcloud_v6.py', dsp_script)
gcloud_cp(DSP_CONFIG_URI, dsp_config)
DSP_SCRIPT_SHA256 = hashlib.sha256(dsp_script.read_bytes()).hexdigest()
DSP_CONFIG_SHA256 = hashlib.sha256(dsp_config.read_bytes()).hexdigest()
sys.path.insert(0, str(dsp_dir))
import adc_to_pointcloud_v6 as adc
if not hasattr(adc, 'build_processor') or not hasattr(adc, 'iter_raw_radar_frames'):
    raise RuntimeError('Pinned DSP script lacks build_processor/iter_raw_radar_frames.')
processor = adc.build_processor(str(dsp_config))

frame_validation_errors = []
if VERIFY_ALL_RAW_FRAME_COUNTS:
    for index, record in enumerate(sorted(sessions.values(), key=lambda r: r['root_uri']), start=1):
        uri = record.get('selected_bin_uri')
        if not uri:
            frame_validation_errors.append({'root_uri': record['root_uri'], 'error': 'missing or ambiguous raw .bin'})
            continue
        local_bin = STAGING_DIR / record['folder_session_id'] / Path(uri).name
        local_bin.parent.mkdir(parents=True, exist_ok=True)
        try:
            gcloud_cp(uri, local_bin)
            frames = sum(1 for _ in adc.iter_raw_radar_frames(local_bin, cfg=processor.cfg, split_multiples=True))
            record['radar_binary_file_bytes'] = local_bin.stat().st_size
            if record.get('bin_format_version') == 3:
                _, payload_bytes = count_chunked_dca_frames(local_bin)
            else:
                payload_bytes = local_bin.stat().st_size
            record['radar_frames_bin'] = frames
            record['radar_payload_bytes_bin'] = payload_bytes
            expected = record.get('radar_frames_metadata')
            if expected is None:
                frame_validation_errors.append({'root_uri': record['root_uri'], 'error': 'metadata has no radar.total_frames'})
            elif frames != expected:
                frame_validation_errors.append({'root_uri': record['root_uri'], 'error': f'frame mismatch: metadata={expected}, bin={frames}'})
            expected_bytes = record.get('radar_bytes_metadata')
            if record.get('bin_format_version') == 3 and expected_bytes is not None and payload_bytes != expected_bytes:
                frame_validation_errors.append({'root_uri': record['root_uri'], 'error': f'payload-byte mismatch: metadata={expected_bytes}, bin={payload_bytes}'})
        except Exception as exc:
            frame_validation_errors.append({'root_uri': record['root_uri'], 'error': repr(exc)})
        finally:
            shutil.rmtree(local_bin.parent, ignore_errors=True)
        if index % 10 == 0 or index == len(sessions):
            print(f'Validated {index:,}/{len(sessions):,} raw bins')

print(f'Raw-frame validation failures: {len(frame_validation_errors):,}')

## 3. Optional exact post-CFAR point count

The raw corpus does not contain a point cloud: it contains raw ADC IQ. When enabled, this pass obtains the production DCA processing script and profile, processes every raw session in a rolling fashion, and sums the processor's `detections_total`. This is the defensible meaning of *radar point count* for the paper, and the configuration/code URI is captured in the evidence. It fails closed: a corpus total is omitted if any session fails.

In [ ]:
point_count_errors = []
if RUN_DETECTION_POINT_COUNT:
    if not hasattr(adc, 'process_session'):
        raise RuntimeError('Pinned DSP script lacks process_session; pin a compatible code revision.')

    for index, record in enumerate(sorted(sessions.values(), key=lambda r: r['root_uri']), start=1):
        uri = record.get('selected_bin_uri')
        if not uri:
            point_count_errors.append({'root_uri': record['root_uri'], 'error': 'missing or ambiguous raw .bin'})
            continue
        session_dir = STAGING_DIR / record['folder_session_id']
        session_dir.mkdir(parents=True, exist_ok=True)
        try:
            gcloud_cp(uri, session_dir / Path(uri).name)
            stats = adc.process_session(processor, session_dir, force=True, quiet=True)
            expected_frames = record.get('radar_frames_bin')
            if not stats.success or stats.frame_errors != 0 or (expected_frames is not None and stats.frames_total != expected_frames):
                raise RuntimeError(f'partial DSP result: frames={stats.frames_total}, expected={expected_frames}, frame_errors={stats.frame_errors}, error={stats.error}')
            record['radar_points_post_cfar'] = int(stats.detections_total)
        except Exception as exc:
            point_count_errors.append({'root_uri': record['root_uri'], 'error': repr(exc)})
        finally:
            shutil.rmtree(session_dir, ignore_errors=True)
        if index % 5 == 0 or index == len(sessions):
            print(f'Point-counted {index:,}/{len(sessions):,} sessions')

print(f'Post-CFAR point-count failures: {len(point_count_errors):,}')


## 4. Fail-closed aggregation, audit artifacts, and paper figures

No aggregate is labelled paper-ready unless the live topology, metadata, uniqueness, raw-frame validation, and (if requested) point-count pass are complete. All results—including zero/missing counts and per-session source values—are written to a single immutable run directory.

In [ ]:
rows = []
for record in sorted(sessions.values(), key=lambda r: (r['dataset_id'], r['folder_session_id'])):
    rows.append({
        'dataset_id': record['dataset_id'],
        'session_id': record.get('metadata_session_id'),
        'folder_session_id': record['folder_session_id'],
        'root_uri': record['root_uri'],
        'recorder': record.get('recorder'),
        'bin_format_version': record.get('bin_format_version'),
        'radar_frames_metadata': record.get('radar_frames_metadata'),
        'radar_frames_bin': record.get('radar_frames_bin'),
        'radar_bytes_metadata': record.get('radar_bytes_metadata'),
        'radar_payload_bytes_bin': record.get('radar_payload_bytes_bin'),
        'radar_binary_file_bytes': record.get('radar_binary_file_bytes'),
        'radar_points_post_cfar': record.get('radar_points_post_cfar'),
        'actual_session_duration_s': record.get('duration_s'),
        'depth_frames_metadata': record.get('depth_frames_metadata'),
        'depth_objects_listed': record.get('depth_object_count'),
        'metadata_sha256': record.get('metadata_sha256'),
        'selected_bin_uri': record.get('selected_bin_uri'),
        'object_count': len(record['objects']),
    })
session_df = pd.DataFrame(rows)

coverage_report = {
    'run_id': RUN_ID,
    'generated_at_utc': datetime.now(timezone.utc).isoformat(),
    'raw_root': RAW_ROOT,
    'gcs_object_count': len(objects),
    'session_root_count': len(sessions),
    'dataset_prefix_count': len({r['dataset_id'] for r in sessions.values()}),
    'unassigned_objects': unassigned_objects,
    'metadata_download_errors': metadata_download_errors,
    'duplicate_metadata_session_ids': {key: [r['root_uri'] for r in value] for key, value in duplicate_metadata_ids.items()},
    'frame_validation_enabled': VERIFY_ALL_RAW_FRAME_COUNTS,
    'frame_validation_errors': frame_validation_errors,
    'point_count_enabled': RUN_DETECTION_POINT_COUNT,
    'point_count_errors': point_count_errors,
    'dsp_code_root': DSP_CODE_ROOT if RUN_DETECTION_POINT_COUNT else None,
    'dsp_config_uri': DSP_CONFIG_URI,
    'dsp_script_sha256': DSP_SCRIPT_SHA256,
    'dsp_config_sha256': DSP_CONFIG_SHA256,
}

# .all()/.nunique()==... on a pandas Series return numpy.bool_, not a native bool. `and`
# returns whichever operand decided the result, so that numpy type would otherwise leak
# straight into aggregate_statistics and break json.dumps (no default=str on the final print).
metadata_complete = bool(len(metadata_download_errors) == 0 and session_df['session_id'].notna().all())
unique_sessions = bool(len(duplicate_metadata_ids) == 0 and session_df['session_id'].nunique() == len(session_df))
topology_complete = bool(len(unassigned_objects) == 0 and session_df['selected_bin_uri'].notna().all())
frame_complete = bool(VERIFY_ALL_RAW_FRAME_COUNTS and len(frame_validation_errors) == 0)
point_complete = bool(RUN_DETECTION_POINT_COUNT and len(point_count_errors) == 0 and session_df['radar_points_post_cfar'].notna().all())
paper_ready = bool(metadata_complete and unique_sessions and topology_complete and frame_complete and point_complete)

aggregate_statistics = {
    'run_id': RUN_ID,
    'generated_at_utc': coverage_report['generated_at_utc'],
    'scope': RAW_ROOT + '/',
    'paper_ready': paper_ready,
    'unique_session_count': int(len(session_df)) if metadata_complete and unique_sessions else None,
    'dataset_prefix_count': int(session_df['dataset_id'].nunique()) if metadata_complete else None,
    'raw_radar_frame_count': int(session_df['radar_frames_bin'].sum()) if frame_complete else None,
    'raw_radar_binary_file_bytes': int(session_df['radar_binary_file_bytes'].sum()) if frame_complete else None,
    'raw_adc_payload_bytes': int(session_df['radar_payload_bytes_bin'].sum()) if frame_complete and (session_df['bin_format_version'] == 3).all() else None,
    'recorded_radar_bytes_metadata': int(session_df['radar_bytes_metadata'].sum()) if session_df['radar_bytes_metadata'].notna().all() else None,
    'recorded_duration_seconds': float(pd.to_numeric(session_df['actual_session_duration_s'], errors='coerce').sum()) if session_df['actual_session_duration_s'].notna().all() else None,
    'depth_frame_count_metadata': int(session_df['depth_frames_metadata'].sum()) if session_df['depth_frames_metadata'].notna().all() else None,
    'depth_objects_listed': int(session_df['depth_objects_listed'].sum()),
    'post_cfar_radar_point_count': int(session_df['radar_points_post_cfar'].sum()) if point_complete else None,
    'post_cfar_point_count_definition': 'Sum of detections_total from pinned adc_to_pointcloud_v6 DCA DSP over each raw session; null unless every session succeeds.',
    'coverage_report': 'coverage_report.json',
    'per_session_table': 'raw_session_statistics.csv',
}

session_csv = WORK_ROOT / 'raw_session_statistics.csv'
coverage_json = WORK_ROOT / 'coverage_report.json'
aggregate_json = WORK_ROOT / 'aggregate_statistics.json'
session_df.to_csv(session_csv, index=False)
coverage_json.write_text(json.dumps(coverage_report, indent=2, default=str) + '\n')
aggregate_json.write_text(json.dumps(aggregate_statistics, indent=2, default=str) + '\n')

for artifact in [WORK_ROOT / 'raw_object_inventory.txt', session_csv, coverage_json, aggregate_json]:
    gcloud_cp(artifact, f'{OUTPUT_ROOT}/{artifact.name}')

print(json.dumps(aggregate_statistics, indent=2))
if not paper_ready:
    raise RuntimeError(f'NOT PAPER-READY. Inspect {OUTPUT_ROOT}/coverage_report.json; aggregate totals intentionally contain nulls where validation is incomplete.')
print(f'Paper-ready evidence uploaded to {OUTPUT_ROOT}/')
